# b4-qlora｜4-bit QLoRA 微調

**目標**：用 4-bit 量化（QLoRA）在 T4 GPU 上微調 Llama-3.2-3B-Instruct

**與 b3-lora 的差異**：
| | b3-lora | b4-qlora |
|---|---|---|
| 模型精度 | bfloat16（全精度） | 4-bit NF4（量化） |
| 記憶體 | ~6GB（MPS） | ~3GB（VRAM） |
| 硬體 | Mac MPS | T4 GPU |
| 額外步驟 | 無 | `prepare_model_for_kbit_training()` |

**執行前確認**：
- Runtime → Change runtime type → **T4 GPU**
- 左側 🔑 Secrets 已設定 `HF_TOKEN`、`WANDB_API_KEY` (需 40 字元以上)
- Google Drive 已有 `MyDrive/Tangram/data/dataset/`

In [ ]:
# ── 確認 GPU ─────────────────────────────────────────────
import torch

assert torch.cuda.is_available(), "GPU 未啟用，請到 Runtime → Change runtime type → T4 GPU"
print(f"GPU：{torch.cuda.get_device_name(0)}")
print(f"VRAM：{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 安裝套件 ──────────────────────────────────────────────
# bitsandbytes 是 4-bit 量化的核心，只能在 CUDA 環境執行（不支援 MPS）
!pip install -q transformers datasets trl peft bitsandbytes accelerate

In [ ]:
# ── 掛載 Google Drive + 讀取 Secrets ─────────────────────
from google.colab import drive, userdata

drive.mount('/content/drive')

HF_TOKEN = userdata.get('HF_TOKEN')
print("HF_TOKEN：", "✓ 已載入" if HF_TOKEN else "✗ 未設定，請到左側 🔑 新增 HF_TOKEN")

In [ ]:
# ── WandB 登入 ────────────────────────────────────────────
import wandb

WANDB_API_KEY = userdata.get('WANDB_API_KEY')
if WANDB_API_KEY:
    if len(WANDB_API_KEY) < 40:
        print(f"WandB：⚠ API Key 長度僅 {len(WANDB_API_KEY)}，可能太短 (需 40+)")
    wandb.login(key=WANDB_API_KEY)
    print("WandB：✓ 已登入")
else:
    print("WandB：✗ 未設定 WANDB_API_KEY，訓練 log 將不會上傳")

In [ ]:
# ── 設定路徑與超參數 ──────────────────────────────────────
from pathlib import Path

MODEL_ID       = "meta-llama/Llama-3.2-3B-Instruct"
DRIVE_BASE     = Path("/content/drive/MyDrive/Tangram")
DATASET_PATH   = DRIVE_BASE / "data/dataset"
CHECKPOINT_DIR = DRIVE_BASE / "checkpoints/b4"
OUTPUT_LOG     = DRIVE_BASE / "outputs/b4_qlora_log.json"
MAX_SEQ_LENGTH = 1024
ALPACA_RATIO   = 0.05

(CHECKPOINT_DIR / "adapter").mkdir(parents=True, exist_ok=True)
OUTPUT_LOG.parent.mkdir(parents=True, exist_ok=True)

print(f"資料集路徑：{DATASET_PATH}")
print(f"資料集存在：{'✓' if DATASET_PATH.exists() else '✗ 請確認已上傳到 Drive'}")

In [ ]:
# ── 載入 tokenizer ────────────────────────────────────────
from transformers import AutoTokenizer

print("載入 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_SEQ_LENGTH
print("完成")

In [ ]:
# ── 載入資料集 + 混入 5% Alpaca ───────────────────────────
from datasets import load_from_disk, Dataset, concatenate_datasets, load_dataset

print("載入 Tangram 資料集...")
dataset  = load_from_disk(str(DATASET_PATH))
train_ds = dataset["train"].select_columns(["text"])
val_ds   = dataset["validation"].select_columns(["text"])
print(f"  訓練集：{len(train_ds)} 筆，驗證集：{len(val_ds)} 筆")

def format_alpaca(sample):
    user_content = sample["instruction"]
    if sample.get("input"):
        user_content += f"\n{sample['input']}"
    messages = [
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": sample["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

alpaca_n   = int(len(train_ds) * ALPACA_RATIO)
print(f"載入 Alpaca（取 {alpaca_n} 筆）...")
alpaca_raw = load_dataset("yahma/alpaca-cleaned", split=f"train[:{alpaca_n}]")
alpaca_ds  = Dataset.from_list([format_alpaca(s) for s in alpaca_raw])
train_mixed = concatenate_datasets([train_ds, alpaca_ds]).shuffle(seed=42)
print(f"  混合後訓練集：{len(train_mixed)} 筆")

In [ ]:
# ── 載入模型（4-bit 量化）與 PEFT 設定 ───────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("載入模型（4-bit 量化）...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)

# QLoRA 必備：處理量化模型的梯度檢查點與 layer norm 精度
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], # 維持與 b3 一致以利比較
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)

# 計算可訓練參數 (DoD 驗證)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
ratio = trainable / total * 100
print("完成")
print(f"VRAM 使用：{torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"全部參數：  {total / 1e9:.3f}B")
print(f"可訓練參數：{trainable / 1e6:.2f}M ({ratio:.3f}%)")

In [ ]:
# ── 訓練設定與執行 ────────────────────────────────────────
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,   # T4 GPU 搭配 4-bit 可以跑 batch 2
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    # 注意：在某些版本 TRL 中，max_seq_length 不在 SFTConfig 而在 SFTTrainer
    # 如果報錯，請移除此行並確認 tokenizer 已設定 model_max_length
    # max_seq_length=MAX_SEQ_LENGTH, 
    dataset_text_field="text",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=42,
    report_to="wandb" if userdata.get('WANDB_API_KEY') else "none",
    run_name="tangram-b4-qlora",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_mixed,
    eval_dataset=val_ds,
    processing_class=tokenizer,
)

print("開始訓練（1 epoch，預計 1.5–2 小時）...")
train_result = trainer.train()
print("\n訓練完成")

In [ ]:
# ── 儲存 adapter + log ────────────────────────────────────
import json

adapter_dir = CHECKPOINT_DIR / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"LoRA adapter 已存至 {adapter_dir}")

log = {
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "target_modules": list(lora_config.target_modules),
    "quantization": "4-bit NF4 + double quant",
    "trainable_params": trainable,
    "trainable_ratio_pct": round(ratio, 4),
    "train_runtime_sec": train_result.metrics.get("train_runtime"),
    "train_loss": train_result.metrics.get("train_loss"),
    "history": trainer.state.log_history,
}
with open(OUTPUT_LOG, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)
print(f"訓練 log 已存至 {OUTPUT_LOG}")

print("\n" + "=" * 50)
print("DoD 驗證")
print("=" * 50)
print(f"  可訓練參數 < 1%  ✓ ({ratio:.4f}%)")
train_losses = [e["loss"] for e in trainer.state.log_history if "loss" in e]
eval_losses  = [e["eval_loss"] for e in trainer.state.log_history if "eval_loss" in e]
print(f"  final train_loss = {train_losses[-1]:.4f}" if train_losses else "  train_loss: 無資料")
print(f"  final eval_loss  = {eval_losses[-1]:.4f}"  if eval_losses  else "  eval_loss: 無資料")

In [ ]:
# ── 下載 log 到本機 ───────────────────────────────────────
from google.colab import files
files.download(str(OUTPUT_LOG))